In [1]:
import pandas as pd
import numpy as np

# 1. Load the data
EXCEL_FILE = '../../output/tyrepressure_tyrewear.xlsx'

try:
    pressure_df = pd.read_excel(EXCEL_FILE, sheet_name='tyre_pressure')
    wear_df = pd.read_excel(EXCEL_FILE, sheet_name='tyre_wear')
except FileNotFoundError:
    raise FileNotFoundError(f"{EXCEL_FILE} not found.")
except ValueError as e:
    raise ValueError("Sheet name not found. Check sheet names.") from e

In [2]:
xls = pd.ExcelFile(EXCEL_FILE)
print("Available sheets:", xls.sheet_names)

pressure_df = pd.read_excel(xls, sheet_name=[s for s in xls.sheet_names if 'pressure' in s.lower()][0])
wear_df = pd.read_excel(xls, sheet_name=[s for s in xls.sheet_names if 'wear' in s.lower()][0])


Available sheets: ['tyre_pressure', 'tyre_wear']


In [3]:
# --- Helper Functions to Flatten the Wide Monorail Data ---

def melt_monorail_data(df, prefix_str, value_name, is_wear=True):
    """
    Converts wide-format train data (ECA1, ICA2, etc.) into long-format for analysis.
    """
    melted_frames = []
    cars = ['eca1', 'ica2', 'ica3', 'eca4']
    bogies = ['bogie1', 'bogie2']
    
    for car in cars:
        for bogie in bogies:
            bogie_sn_col = f'{prefix_str}.{car}.{bogie}.bogie_sn'
            if bogie_sn_col not in df.columns:
                continue
            
            # Base identifier columns
            base_cols = ['workorder_id', 'functional_location', f'{prefix_str}.approval_date', bogie_sn_col]
            
            # Identify measurement columns for this specific bogie
            prefix = f'{prefix_str}.{car}.{bogie}.'
            meas_cols = [c for c in df.columns if c.startswith(prefix) and c != bogie_sn_col]
            
            temp_df = df[base_cols + meas_cols].copy()
            temp_df.rename(columns={
                bogie_sn_col: 'bogie_sn',
                f'{prefix_str}.approval_date': 'date',
                'functional_location': 'train_id'
            }, inplace=True)
            temp_df['car'] = car
            temp_df['bogie_id'] = bogie
            
            # Melt the measurements
            temp_melted = temp_df.melt(
                id_vars=['workorder_id', 'train_id', 'date', 'bogie_sn', 'car', 'bogie_id'],
                var_name='raw_label',
                value_name=value_name
            )
            
            # Parse the column names to get Wheel Type and Groove/Pressure Type
            # Example: "tyre_wear.eca1.bogie1.top_guide_wheel.a.groove1"
            temp_melted['wheel_type'] = temp_melted['raw_label'].apply(lambda x: x.split('.')[-3] if 'load_wheel' not in x else 'load_wheel')
            temp_melted['wheel_pos'] = temp_melted['raw_label'].apply(lambda x: x.split('.')[-2])
            temp_melted['sub_type'] = temp_melted['raw_label'].apply(lambda x: x.split('.')[-1])
            
            melted_frames.append(temp_melted)
            
    return pd.concat(melted_frames, ignore_index=True)

# 2. Process and Clean the Data
print("Processing data...")
wear_long = melt_monorail_data(wear_df, 'tyre_wear', 'depth', is_wear=True)
pressure_long = melt_monorail_data(pressure_df, 'tyre_pressure', 'pressure', is_wear=False)

# Clean depth: Handle 'new' tyres and non-numeric values
wear_long['depth'] = pd.to_numeric(wear_long['depth'].replace('new', 10.0), errors='coerce')
wear_long['date'] = pd.to_datetime(wear_long['date'], format='mixed', dayfirst=True)

# Clean pressure
pressure_long['pressure'] = pd.to_numeric(pressure_long['pressure'], errors='coerce')
pressure_long['date'] = pd.to_datetime(pressure_long['date'], format='mixed', dayfirst=True)

# 3. Calculate Wear Rates (Predictive Feature)
wear_long = wear_long.sort_values(['bogie_sn', 'wheel_type', 'wheel_pos', 'sub_type', 'date'])
wear_long['prev_depth'] = wear_long.groupby(['bogie_sn', 'wheel_type', 'wheel_pos', 'sub_type'])['depth'].shift(1)
wear_long['prev_date'] = wear_long.groupby(['bogie_sn', 'wheel_type', 'wheel_pos', 'sub_type'])['date'].shift(1)
wear_long['days_diff'] = (wear_long['date'] - wear_long['prev_date']).dt.days

# Only look at wear deltas (ignore replacements where depth increases)
wear_long['wear_delta'] = wear_long['prev_depth'] - wear_long['depth']
wear_rate_df = wear_long[(wear_long['wear_delta'] > 0) & (wear_long['days_diff'] > 0)].copy()
wear_rate_df['mm_per_day'] = wear_rate_df['wear_delta'] / wear_rate_df['days_diff']

# 4. Calculate Pressure Drops (Leak Detection Feature)
# Pivot before/after pressure into two columns
pressure_pivot = pressure_long.pivot_table(
    index=['workorder_id', 'train_id', 'date', 'bogie_sn', 'car', 'bogie_id', 'wheel_type', 'wheel_pos'],
    columns='sub_type', values='pressure'
).reset_index()

pressure_pivot = pressure_pivot.sort_values(['bogie_sn', 'wheel_type', 'wheel_pos', 'date'])
pressure_pivot['prev_after'] = pressure_pivot.groupby(['bogie_sn', 'wheel_type', 'wheel_pos'])['after'].shift(1)
pressure_pivot['pressure_drop'] = pressure_pivot['prev_after'] - pressure_pivot['before']

# 5. Remaining Useful Life (RUL) Prediction Logic
# Group by wheel to find average wear rate and last known depth
rul_df = wear_rate_df.groupby(['bogie_sn', 'wheel_type', 'wheel_pos']).agg({
    'mm_per_day': 'mean',
    'depth': 'last'
}).reset_index()

# Assuming safety replacement threshold is 3.0mm
SAFETY_LIMIT = 3.0
rul_df['days_to_replacement'] = (rul_df['depth'] - SAFETY_LIMIT) / rul_df['mm_per_day']

# 6. Save Results
# rul_df.to_csv('tyre_replacement_forecast.csv', index=False)
# pressure_pivot.to_csv('tyre_leak_analysis.csv', index=False)

print("Analysis Complete! Files saved:")
print("- tyre_replacement_forecast.csv (Predictions)")
print("- tyre_leak_analysis.csv (Anomaly Detection)")

Processing data...


DateParseError: Unknown datetime string format, unable to parse: 13/NA/NA, at position 439